In [1]:
import pandas as pd
import sys

sys.path.append('..')
from src.features.text_features import (
    build_pair_features,
    get_feature_columns,
)

In [2]:
# ============================================================
# LOAD PAIRS
# ============================================================

train_pairs = pd.read_parquet("../data/processed/train_pairs_augmented_v2.parquet")
valid_pairs = pd.read_parquet("../data/processed/valid_pairs.parquet")
test_pairs = pd.read_parquet("../data/processed/test_pairs.parquet")

print(f"Train: {len(train_pairs):,}")
print(f"Valid: {len(valid_pairs):,}")
print(f"Test:  {len(test_pairs):,}")

Train: 180,894
Valid: 15,306
Test:  14,812


In [3]:
# ============================================================
# BUILD FEATURES
# ============================================================

train_features = build_pair_features(train_pairs)
valid_features = build_pair_features(valid_pairs)
test_features = build_pair_features(test_pairs)

feature_cols = get_feature_columns()

print(f"Feature count: {len(feature_cols)}")
print(feature_cols)

Feature count: 26
['fuzz_ratio', 'partial_ratio', 'token_sort_ratio', 'token_set_ratio', 'exact_match', 'token_jaccard', 'common_tokens', 'len_1', 'len_2', 'length_diff', 'token_count_1', 'token_count_2', 'token_count_diff', 'has_legal_form_1', 'has_legal_form_2', 'same_legal_form', 'core_fuzz_ratio', 'core_token_sort_ratio', 'core_token_set_ratio', 'core_exact_match', 'same_script', 'same_relation_kind', 'same_relation_role', 'is_garbage_1', 'is_garbage_2', 'has_garbage_entity']


In [4]:
# ============================================================
# FEATURE PREVIEW
# ============================================================

display(
    train_features[
        feature_cols + ["label", "pair_type"]
    ].head()
)

,fuzz_ratio,partial_ratio,token_sort_ratio,token_set_ratio,exact_match,token_jaccard,common_tokens,len_1,len_2,length_diff,...,core_token_set_ratio,core_exact_match,same_script,same_relation_kind,same_relation_role,is_garbage_1,is_garbage_2,has_garbage_entity,label,pair_type
0,47.457627,49.056604,40.677966,40.677966,0,0.00,0,28,31,3,...,40.677966,0,1,1,1,0,0,0,0,easy_negative_different_public_id
1,83.333333,100.000000,83.333333,100.000000,0,0.75,3,15,21,6,...,100.000000,0,1,1,1,0,0,0,0,augmented_hard_negative
2,67.924528,100.000000,67.924528,100.000000,0,0.75,3,35,18,17,...,100.000000,0,1,0,0,0,0,0,1,augmented_positive
3,35.483871,43.137255,38.709677,38.709677,0,0.00,0,31,31,0,...,38.709677,0,0,0,0,0,0,0,0,easy_negative_different_public_id
4,100.000000,100.000000,100.000000,100.000000,1,1.00,4,31,31,0,...,100.000000,1,1,1,1,0,0,0,1,positive_same_public_id


In [5]:
# ============================================================
# FEATURE SUMMARY
# ============================================================

display(
    train_features[
        feature_cols
    ].describe().T
)

,count,mean,std,min,25%,50%,75%,max
fuzz_ratio,180894.0,79.394028,23.035816,5.405405,74.074074,90.322581,98.461538,100.0
partial_ratio,180894.0,85.870554,21.970879,10.000000,89.655172,100.000000,100.000000,100.0
token_sort_ratio,180894.0,79.629207,23.585429,5.000000,74.418605,90.322581,100.000000,100.0
token_set_ratio,180894.0,83.754007,24.769609,5.000000,81.481481,96.551724,100.000000,100.0
exact_match,180894.0,0.246227,0.430814,0.000000,0.000000,0.000000,0.000000,1.0
token_jaccard,180894.0,0.505119,0.392940,0.000000,0.200000,0.500000,1.000000,1.0
common_tokens,180894.0,1.772906,1.272884,0.000000,1.000000,2.000000,3.000000,7.0
len_1,180894.0,21.522184,7.586379,5.000000,15.000000,17.000000,29.000000,63.0
len_2,180894.0,22.424154,7.057063,4.000000,16.000000,22.000000,28.000000,63.0
length_diff,180894.0,3.543561,4.488272,0.000000,0.000000,1.000000,6.000000,53.0


In [6]:
# ============================================================
# MISSING FEATURE CHECK
# ============================================================

missing = (
    train_features[feature_cols]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

display(missing[missing > 0])

Series([], dtype: int64)

In [7]:
# ============================================================
# FEATURE DISTRIBUTION BY LABEL
# ============================================================

important_features = [
    "fuzz_ratio",
    "token_sort_ratio",
    "token_set_ratio",
    "core_token_set_ratio",
    "token_jaccard",
    "length_diff",
    "same_country",
    "same_company_public_id",
]

for col in important_features:
    print("=" * 60)
    print(col)
    print(
        train_features
        .groupby("label")[col]
        .describe()
    )

fuzz_ratio
          count       mean        std        min        25%         50%  \
label                                                                     
0      115619.0  70.482776  23.321087   5.405405  47.619048   81.481481   
1       65275.0  95.178172  10.836580  29.629630  97.297297  100.000000   

              75%    max  
label                     
0       90.322581  100.0  
1      100.000000  100.0  
token_sort_ratio
          count       mean        std        min         25%         50%  \
label                                                                      
0      115619.0  69.959891  23.851204   5.000000   43.076923   81.481481   
1       65275.0  96.756081   8.660698  25.925926  100.000000  100.000000   

              75%    max  
label                     
0       90.322581  100.0  
1      100.000000  100.0  
token_set_ratio
          count       mean        std        min         25%         50%  \
label                                                     

In [8]:
# ============================================================
# SAVE FEATURE DATASETS
# ============================================================

train_features.to_parquet(
    "../data/processed/train_features.parquet",
    index=False,
)

valid_features.to_parquet(
    "../data/processed/valid_features.parquet",
    index=False,
)

test_features.to_parquet(
    "../data/processed/test_features.parquet",
    index=False,
)

print("Saved:")
print("../data/processed/train_features.parquet")
print("../data/processed/valid_features.parquet")
print("../data/processed/test_features.parquet")

Saved:
../data/processed/train_features.parquet
../data/processed/valid_features.parquet
../data/processed/test_features.parquet
